In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.fft
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from gnlse import plot_intensity

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [5]:
data1 = np.load('./data/total_fields_3.npy')
data2 = np.load('./data/total_fields_4.npy')
data3 = np.load('./data/total_fields_1.npy')
total_data = np.concatenate([data1, data2, data3], axis=0)
print(total_data.shape)
print(total_data.dtype)

(48, 21, 64, 64, 2048)
complex64


In [4]:
total_data = np.load('../neuraloperator/total_data.npy')
print(total_data.shape)
print(total_data.dtype)

(48, 21, 48, 48, 1024)
complex64


In [5]:
core_radius = 16.0e-6 / 2
Lx, Ly = 4 * core_radius, 4 * core_radius
Nx, Ny = 48, 48
Nt = 1024
extent = [-Lx/2, Lx/2, -Ly/2, Ly/2]

In [8]:
test_fields_input = total_data[-1][0]
test_fields_output = total_data[-1][-1]
plot_intensity(test_fields_output)

In [6]:
class SpectralConv3d(nn.Module):
    def __init__(self, in_channels, out_channels, modes, bias=True):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        # modes = (modes_x, modes_y, modes_t)
        self.modes_x, self.modes_y, self.modes_t = modes
        scale = 1 / (in_channels * out_channels)
        self.weight = nn.Parameter(scale * torch.randn(in_channels, out_channels,
                                                       self.modes_x, self.modes_y, self.modes_t, dtype=torch.cfloat))
        self.bias = nn.Parameter(torch.zeros(out_channels)) if bias else None

    def compl_mul3d(self, input, weights):
        # input: (B, in_c, kx, ky, kt)
        # weights: (in_c, out_c, kx, ky, kt)
        return torch.einsum("bixyz,ioxyz->boxyz", input, weights)

    def forward(self, x):
        B, C, X, Y, T = x.shape
        x = torch.fft.fftn(x, dim=(-3, -2, -1))  # 3D FFT over (x, y, t)
        out = torch.zeros(B, self.out_channels, X, Y, T, dtype=torch.cfloat, device=x.device)

        # Only keep low-frequency modes
        out[:, :, :self.modes_x, :self.modes_y, :self.modes_t] = \
            self.compl_mul3d(x[:, :, :self.modes_x, :self.modes_y, :self.modes_t],
                             self.weight)

        # iFFT back to real space
        out = torch.fft.ifftn(out, dim=(-3, -2, -1))
        if self.bias is not None:
            out = out + self.bias.view(1, -1, 1, 1, 1)
        return out.real  # real tensor for next layers (split real/imag separately later if needed)


# ========================
# 3D FNO Block
# ========================
class FNO3DBlock(nn.Module):
    def __init__(self, in_channels, out_channels, modes=(12, 12, 12), width=64):
        super().__init__()
        self.spectral_conv = SpectralConv3d(in_channels, out_channels, modes)
        self.pointwise_conv = nn.Conv3d(in_channels, out_channels, 1)
        self.activation = nn.GELU()

    def forward(self, x):
        out = self.spectral_conv(x) + self.pointwise_conv(x)
        return x + self.activation(out)
        # return self.activation(out)

# ========================
# 3D FNO Model
# ========================
class FNO3D(nn.Module):
    def __init__(self, in_channels=2, out_channels=2, width=32, n_layers=4,
                 modes=(12, 12, 12)):
        super().__init__()
        self.input_proj = nn.Conv3d(in_channels, width, 1)
        self.fno_layers = nn.ModuleList([
            FNO3DBlock(width, width, modes, width) for _ in range(n_layers)
        ])
        self.output_proj = nn.Sequential(
            nn.Conv3d(width, width // 2, 1),
            nn.GELU(),
            nn.Conv3d(width // 2, out_channels, 1)
        )

    def forward(self, x):
        # x: (B, 2, X, Y, T)
        x = self.input_proj(x)
        for layer in self.fno_layers:
            x = layer(x)
        return self.output_proj(x)

In [7]:
# Hyperparameters
lr = 0.01
batch_size = 8
epochs = 100
width = 16
num_layers = 4
mode_x, mode_y, mode_t = 16, 16, 16

In [8]:
model = FNO3D(
    in_channels=2,
    out_channels=2,
    width=width,
    n_layers=num_layers,
    modes=(mode_x, mode_y, mode_t),
).to(device)


# Print the number of tunable parameters
print('Number of parameters : ',flush=True)
print(sum(p.numel() for p in model.parameters() if p.requires_grad))

Number of parameters : 
4195658


In [9]:
optimizer = optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
loss_fn = nn.MSELoss()

In [11]:
def train_one_epoch(model, data, batch_size, optimizer, loss_fn, device, eps=1e-8):
    """
    Train for one epoch: predict next z-step from current z-step
    
    Args:
        model: FNO3D model
        data: (n_samples, n_z_steps, 2, Nx, Ny, Nt)
        batch_size: batch size
        optimizer: optimizer
        loss_fn: loss function
        device: device
        eps: epsilon for normalization
    
    Returns:
        average loss for the epoch
    """
    model.train()
    
    n_samples = data.shape[0]
    n_z_steps = data.shape[1]
    
    # Create all (z_i -> z_i+1) pairs
    # For each sample, we have (n_z_steps - 1) training pairs
    all_pairs = []
    
    for sample_idx in range(n_samples):
        for z_idx in range(n_z_steps - 1):
            input_field = data[sample_idx, z_idx]      # (2, Nx, Ny, Nt)
            target_field = data[sample_idx, z_idx + 1]  # (2, Nx, Ny, Nt)
            all_pairs.append((input_field, target_field))
    
    # Shuffle pairs for better training
    indices = torch.randperm(len(all_pairs))
    
    total_loss = 0
    num_batches = 0
    
    for i in range(0, len(all_pairs), batch_size):
        batch_indices = indices[i:i+batch_size]
        
        # Gather batch
        batch_inputs = []
        batch_targets = []
        
        for idx in batch_indices:
            inp, tgt = all_pairs[idx]
            batch_inputs.append(inp)
            batch_targets.append(tgt)
        
        batch_inputs = torch.stack(batch_inputs).to(device)    # (B, 2, Nx, Ny, Nt)
        batch_targets = torch.stack(batch_targets).to(device)  # (B, 2, Nx, Ny, Nt)
        
        optimizer.zero_grad()
        
        # Forward pass
        predictions = model(batch_inputs)
        
        # Compute loss
        loss = loss_fn(predictions, batch_targets)
        loss = loss / (torch.mean(batch_targets**2) + eps)
        
        total_loss += loss.item()
        num_batches += 1
        
        # Backward pass
        loss.backward()
        optimizer.step()
        scheduler.step()
    
    return total_loss / num_batches


# ========================
# Test Function
# ========================
def test_one_epoch(model, data, batch_size, loss_fn, device, eps=1e-8):
    """
    Test for one epoch: predict next z-step from current z-step
    
    Args:
        model: FNO3D model
        data: (n_samples, n_z_steps, 2, Nx, Ny, Nt)
        batch_size: batch size
        loss_fn: loss function
        device: device
        eps: epsilon for normalization
    
    Returns:
        average loss for the epoch
    """
    model.eval()
    
    n_samples = data.shape[0]
    n_z_steps = data.shape[1]
    
    # Create all (z_i -> z_i+1) pairs
    all_pairs = []
    
    for sample_idx in range(n_samples):
        for z_idx in range(n_z_steps - 1):
            input_field = data[sample_idx, z_idx]
            target_field = data[sample_idx, z_idx + 1]
            all_pairs.append((input_field, target_field))
    
    total_loss = 0
    num_batches = 0
    
    with torch.no_grad():
        for i in range(0, len(all_pairs), batch_size):
            # Gather batch
            batch_inputs = []
            batch_targets = []
            
            for j in range(i, min(i + batch_size, len(all_pairs))):
                inp, tgt = all_pairs[j]
                batch_inputs.append(inp)
                batch_targets.append(tgt)
            
            batch_inputs = torch.stack(batch_inputs).to(device)
            batch_targets = torch.stack(batch_targets).to(device)
            
            # Forward pass
            predictions = model(batch_inputs)
            
            # Compute loss
            loss = loss_fn(predictions, batch_targets)
            loss = loss / (torch.mean(batch_targets**2) + eps)
            
            total_loss += loss.item()
            num_batches += 1
    
    return total_loss / num_batches

In [12]:
num_samples = total_data.shape[0]
n_z_steps = total_data.shape[1]
print(num_samples, n_z_steps)

48 21


In [13]:
n_train = int(num_samples * 0.8)
n_test = num_samples - n_train
print(f'Train samples: {n_train}, Test samples: {n_test}')
print(f'Z-steps per sample: {n_z_steps}')

train_data = total_data[:n_train]
test_data = total_data[n_train:]

train_data = np.reshape(train_data, (n_train, n_z_steps, 1, Nx, Ny, Nt))
test_data = np.reshape(test_data, (n_test, n_z_steps, 1, Nx, Ny, Nt))
train_data = torch.tensor(np.concatenate([train_data.real, train_data.imag], axis=2))
test_data = torch.tensor(np.concatenate([test_data.real, test_data.imag], axis=2))

print(train_data.shape)

Train samples: 38, Test samples: 10
Z-steps per sample: 21
torch.Size([38, 21, 2, 48, 48, 1024])


In [14]:
train_loss_list = []
test_loss_list = []

In [ ]:
eps = 1e-8
for epoch in range(epochs):
    # Train
    train_loss = train_one_epoch(model, train_data, batch_size, optimizer, loss_fn, device, eps)
    
    # Test
    test_loss = test_one_epoch(model, test_data, batch_size, loss_fn, device, eps)
    
    train_loss_list.append(train_loss)
    test_loss_list.append(test_loss)
    
    print(f"Epoch {epoch+1:3d}/{epochs} | Train Loss: {train_loss:.6f} | Test Loss: {test_loss:.6f}", flush=True)
    
    # Save checkpoint
    if (epoch + 1) % 10 == 0:
        checkpoint_path = f"checkpoint_epoch_{epoch+1}.pth"
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'test_loss': test_loss,
        }, checkpoint_path)
        print(f"  Saved checkpoint: {checkpoint_path}")

Epoch   1/100 | Train Loss: 0.824652 | Test Loss: 0.856002
Epoch   2/100 | Train Loss: 0.787855 | Test Loss: 0.872945
Epoch   3/100 | Train Loss: 0.766636 | Test Loss: 0.830523
Epoch   4/100 | Train Loss: 0.720686 | Test Loss: 0.816736
Epoch   5/100 | Train Loss: 0.660262 | Test Loss: 0.727437
Epoch   6/100 | Train Loss: 0.554177 | Test Loss: 0.772850
Epoch   7/100 | Train Loss: 0.533050 | Test Loss: 0.623560
Epoch   8/100 | Train Loss: 0.426293 | Test Loss: 0.610380
Epoch   9/100 | Train Loss: 0.449802 | Test Loss: 0.588401
Epoch  10/100 | Train Loss: 0.361089 | Test Loss: 0.564304
  Saved checkpoint: checkpoint_epoch_10.pth
Epoch  11/100 | Train Loss: 0.389705 | Test Loss: 0.545619
Epoch  12/100 | Train Loss: 0.319152 | Test Loss: 0.506051
Epoch  13/100 | Train Loss: 0.342932 | Test Loss: 0.521408
Epoch  14/100 | Train Loss: 0.295206 | Test Loss: 0.482788
Epoch  15/100 | Train Loss: 0.306218 | Test Loss: 0.499846
Epoch  16/100 | Train Loss: 0.275747 | Test Loss: 0.457592
Epoch  17/10

In [11]:
ckpt_path = './checkpoint_epoch_70.pth'
model.load_state_dict(torch.load(ckpt_path, weights_only=True)['model_state_dict'])
model.eval()

FNO3D(
  (input_proj): Conv3d(2, 16, kernel_size=(1, 1, 1), stride=(1, 1, 1))
  (fno_layers): ModuleList(
    (0-3): 4 x FNO3DBlock(
      (spectral_conv): SpectralConv3d()
      (pointwise_conv): Conv3d(16, 16, kernel_size=(1, 1, 1), stride=(1, 1, 1))
      (activation): GELU(approximate='none')
    )
  )
  (output_proj): Sequential(
    (0): Conv3d(16, 8, kernel_size=(1, 1, 1), stride=(1, 1, 1))
    (1): GELU(approximate='none')
    (2): Conv3d(8, 2, kernel_size=(1, 1, 1), stride=(1, 1, 1))
  )
)

In [25]:
test = test_data[0]
print(test.shape)

torch.Size([21, 2, 64, 64, 2048])


In [26]:
test_input_field = test[0][0] + 1j*test[0][1]
print(test_input_field.shape)

torch.Size([64, 64, 2048])


In [27]:
plot_intensity(test_input_field)

TypeError: sum() received an invalid combination of arguments - got (out=NoneType, axis=int, ), but expected one of:
 * (*, torch.dtype dtype = None)
      didn't match because some of the keywords were incorrect: out, axis
 * (tuple of ints dim, bool keepdim = False, *, torch.dtype dtype = None)
 * (tuple of names dim, bool keepdim = False, *, torch.dtype dtype = None)
